<a href="https://colab.research.google.com/github/ShaheemJ/CelestAI/blob/main/Actual_256x256_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install tqdm pillow

In [ ]:
!pip install datasets --quiet

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp
from torch import autograd
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

%pylab inline
from datasets import load_dataset

In [ ]:
# Set seeds for reproducibility
def seed_everything(seed=33):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(seed=614)

In [ ]:
# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Configuration
CONFIG = {
    'BATCH_SIZE'        : 128,
    'noise_dim'         : 128,
    'input_size'        : 256,
    'dp_rate'           : 0.3,
    'gauss_std'         : 0.1,
    'std_decay_rate'    : 0.,  # 1/100 was commented out
    'guassian_noise'    : ['discriminator'],
    'nthreads'          : 2,
    'max_lr'            : 2e-4,
    'betas'             : (0.5, 0.999),
    'seed'              : 614,
    'use_amp'           : True,
    'log_interval'      : 10,
    'num_epochs'        : 50,
    'save_interval'     : 5,
    'sample_size'       : 16,  # Number of images to generate for visualization
    'checkpoint_dir'    : './checkpoints',
    'sample_dir'        : './samples'
}

# Create directories for checkpoints and samples
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['sample_dir'], exist_ok=True)

In [ ]:
import itertools
# Preview dataset samples
def preview_dataset():
    # Load a small sample to preview
    dset_ls = load_dataset("MultimodalUniverse/gz10", streaming=True, split='train').with_format("numpy")
    subset = list(itertools.islice(dset_ls, 4))

    # Plot 4 images
    fig, axes = plt.subplots(1, 4, figsize=(12, 5))

    for i in range(4):
        example = subset[i]
        image = np.array(example['rgb_image'])

        axes[i].imshow(image)
        axes[i].axis('off')

        title = f"Image {example['object_id']}" if 'object_id' in example else f"Image {i}"
        axes[i].set_title(title)

        print(f"RGB Image {i} Dimensions:", image.shape)

    plt.tight_layout()
    plt.show()

    print("Dataset keys:", example.keys())

In [ ]:
# Data preparation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

class gz10Dataset(Dataset):
    def __init__(self, split='train', transform=None):
        """
        Initializes the gz10Dataset.

        Args:
            split (str): Which split of the dataset to load ('train', 'test', etc.).
            transform (callable, optional): Optional transform to be applied on an image sample.
        """
        # Load the dataset from Hugging Face
        self.dataset = load_dataset("MultimodalUniverse/gz10", split=split)
        self.transform = transform

    def __len__(self):
        """
        Returns the number of samples in the dataset.
        """
        return len(self.dataset)

    def __getitem__(self, idx):
        """
        Retrieves the sample at the given index.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            tuple: (image, label) where image is transformed (if a transform is provided)
                   and label is the integer classification (0-9) from the 'gz10_label' field.
        """
        sample = self.dataset[idx]

        # Get the image and label using the correct keys
        image = sample['rgb_image']
        label = sample['gz10_label']

        # Convert image to PIL format if it's a NumPy array
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)

        # Apply the transformation if provided
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# Create DataLoaders with custom train/validation split
def get_data_loaders(batch_size, nthreads):
    # Create dataset
    full_dataset = gz10Dataset(split='train', transform=transform)

    # Split dataset into train and validation
    dataset_size = len(full_dataset)
    val_size = int(dataset_size * 0.1)  # 10% for validation
    train_size = dataset_size - val_size

    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(CONFIG['seed'])  # For reproducibility
    )

    print(f"Dataset size: {dataset_size}, Train: {train_size}, Validation: {val_size}")

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=nthreads,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=nthreads,
        pin_memory=True,
        drop_last=False
    )

    return train_loader, val_loader

In [ ]:
# Model Definitions
class GaussianNoise(nn.Module):
    def __init__(self, std=0.1, decay_rate=0):
        super().__init__()
        self.std = std
        self.decay_rate = decay_rate

    def decay_step(self):
        self.std = max(self.std - self.decay_rate, 0)

    def forward(self, x):
        if self.training:
            return x + torch.empty_like(x).normal_(std=self.std)
        else:
            return x

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim, activation=nn.LeakyReLU, dp_rate=0.3):
        super().__init__()
        self.activation = activation
        self.stem = nn.Sequential(OrderedDict([
            ('linear',  nn.Linear(noise_dim, 1024*4*4, bias=False)),
            ('bn',      nn.BatchNorm1d(1024*4*4)),
            ('act',     activation(0.2, inplace=True)),
            ('dropout', nn.Dropout(dp_rate)),
        ]))

        self.stacks = nn.Sequential(
            self.upsample(1024, 512, dp_rate=dp_rate),
            self.upsample(512, 256, dp_rate=dp_rate),
            self.upsample(256, 128, dp_rate=dp_rate),
            self.upsample(128, 64, dp_rate=dp_rate),
            self.upsample(64, 32, dp_rate=0)
        )

        self.gen = nn.Sequential(OrderedDict([
            ('conv',    nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)),
            ('act',     nn.Tanh()),
        ]))

    def upsample(self, in_channels, out_channels, bn=True, dp_rate=0.3):
        layers = [nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, bias=not bn, padding=1)]
        if bn:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(self.activation(0.2, inplace=True))
        if dp_rate > 0:
            layers.append(nn.Dropout2d(dp_rate))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = x.view(-1, 1024, 4, 4)
        x = self.stacks(x)
        x = self.gen(x)
        return x

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, activation=nn.LeakyReLU, std=0.1, std_decay_rate=0):
        super().__init__()
        self.std = std
        self.std_decay_rate = std_decay_rate
        self.activation = activation
        self.stacks = nn.Sequential(
            self.downsample(3, 32, bn=False),  # Input channels specified
            self.downsample(32, 64),
            self.downsample(64, 128),
            self.downsample(128, 256),
            self.downsample(256, 512),
            self.downsample(512, 1024),
        )

        self.head = nn.Sequential(OrderedDict([
            ('gauss', GaussianNoise(self.std, self.std_decay_rate)),
            ('linear', nn.Linear(1024 * 4 * 4, 1)),  # Calculate input size
            # No sigmoid as we're using BCEWithLogitsLoss
        ]))

    def downsample(self, in_channels, out_channels, bn=True, stride=2):
        layers = [
            GaussianNoise(self.std, self.std_decay_rate),
            nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=stride, bias=not bn, padding=1)  # Changed from LazyConv2d
        ]
        if bn:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(self.activation(0.2, inplace=True))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stacks(x)
        x = x.flatten(1)
        x = self.head(x)
        return x

In [ ]:
# Initialize weights
@torch.no_grad()
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
# Helper for tracking metrics
class AverageMeter:
    def __init__(self, name=None):
        self.name = name
        self.reset()

    def reset(self):
        self.val = self.sum = self.count = self.avg = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

# Set gradients helper function
def set_grads(grads, params):
    for g, p in zip(grads, params):
        p.grad = g

In [ ]:
# Save samples from generator
def save_samples(epoch, netG, device, sample_size=16, noise_dim=128):
    netG.eval()
    with torch.no_grad():
        # Generate fixed noise for consistent samples
        fixed_noise = torch.randn(sample_size, noise_dim, device=device)
        fake_images = netG(fixed_noise)

        # Convert to numpy and denormalize
        fake_images = fake_images.detach().cpu().numpy()
        # Move channel dim to last position and denormalize from [-1, 1] to [0, 1]
        fake_images = np.transpose(fake_images, (0, 2, 3, 1))
        fake_images = (fake_images + 1) / 2.0
        fake_images = np.clip(fake_images, 0, 1)

        # Plot and save
        plt.figure(figsize=(16, 16))
        for i in range(sample_size):
            plt.subplot(int(np.sqrt(sample_size)), int(np.sqrt(sample_size)), i+1)
            plt.imshow(fake_images[i])
            plt.axis('off')

        plt.tight_layout()
        plt.savefig(f"{CONFIG['sample_dir']}/epoch_{epoch}.png")
        plt.close()

In [ ]:
def train_epoch(train_loader, netG, netD, optG, optD, scaler, criterion, noise_dim, epoch=1, use_amp=True, log_interval=10, device=DEVICE):
    netG.train()
    netD.train()
    lossesG = AverageMeter()
    lossesD = AverageMeter()

    with tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch:>2}") as pbar:
        for idx, (real_images, _) in pbar:  # Unpack correctly
            # Move data to device
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            # Generate noise for the generator
            noise = torch.randn(batch_size, noise_dim, device=device)

            # ===== Train Discriminator =====
            optD.zero_grad(set_to_none=True)

            # Train with mixed precision
            with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', enabled=use_amp):
                # Generate fake images
                with torch.no_grad():  # Don't track gradients for G when updating D
                    fake_images = netG(noise)

                # Get outputs
                fake_out = netD(fake_images)
                real_out = netD(real_images)

                # Label smoothing - soft and noisy real labels
                real_labels = torch.empty_like(real_out).uniform_(0.9, 1.0)
                fake_labels = torch.empty_like(fake_out).uniform_(0.0, 0.1)

                # Calculate losses
                lossD_real = criterion(real_out, real_labels)
                lossD_fake = criterion(fake_out, fake_labels)
                lossD = lossD_real + lossD_fake

            # Backward pass with scaling
            scaler.scale(lossD).backward()
            scaler.step(optD)
            scaler.update()

            # ===== Train Generator =====
            optG.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', enabled=use_amp):
                # Generate fake images again, this time with gradients
                fake_images = netG(noise)
                fake_out = netD(fake_images)

                # Generator wants discriminator to think fake images are real
                lossG = criterion(fake_out, torch.ones_like(fake_out))

            # Backward pass with scaling
            scaler.scale(lossG).backward()
            scaler.step(optG)
            scaler.update()

            # Update statistics
            lossesG.update(lossG.item(), batch_size)
            lossesD.update(lossD.item(), batch_size)

            # Update progress bar
            if idx % log_interval == 0:
                pbar.set_postfix({
                    'G_loss': f"{lossesG.avg:.4f}",
                    'D_loss': f"{lossesD.avg:.4f}"
                })

    return lossesG.avg, lossesD.avg

In [ ]:
# Main training function
def train(netG, netD, train_loader, criterion, optG, optD, scaler, config, device=DEVICE):
    # Training loop
    G_losses = []
    D_losses = []

    for epoch in range(1, config['num_epochs'] + 1):
        # Train for one epoch
        g_loss, d_loss = train_epoch(
            train_loader, netG, netD, optG, optD, scaler, criterion,
            config['noise_dim'], epoch, config['use_amp'],
            config['log_interval'], device
        )

        # Record losses
        G_losses.append(g_loss)
        D_losses.append(d_loss)

        # Save samples
        if epoch % 1 == 0:
            save_samples(epoch, netG, device, config['sample_size'], config['noise_dim'])

        # Save model checkpoints
        if epoch % config['save_interval'] == 0:
            torch.save({
                'epoch': epoch,
                'generator_state_dict': netG.state_dict(),
                'discriminator_state_dict': netD.state_dict(),
                'generator_optimizer': optG.state_dict(),
                'discriminator_optimizer': optD.state_dict(),
                'G_losses': G_losses,
                'D_losses': D_losses,
            }, f"{config['checkpoint_dir']}/checkpoint_epoch_{epoch}.pt")

    # Plot loss curves
    plt.figure(figsize=(10, 5))
    plt.plot(G_losses, label='Generator Loss')
    plt.plot(D_losses, label='Discriminator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f"{config['checkpoint_dir']}/loss_curves.png")
    plt.show()

    return G_losses, D_losses

In [ ]:
def setup_training(netG, netD, config):
    # Initialize optimizers
    optG = torch.optim.Adam(netG.parameters(), lr=config['max_lr'], betas=config['betas'])
    optD = torch.optim.Adam(netD.parameters(), lr=config['max_lr'], betas=config['betas'])

    # Initialize loss function
    criterion = nn.BCEWithLogitsLoss()

    # Initialize AMP scaler with updated API
    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
    scaler = torch.amp.GradScaler(device=device_type, enabled=config['use_amp'])

    return optG, optD, criterion, scaler

In [ ]:
preview_dataset()

In [ ]:
import itertools

# Create data loaders
train_loader, val_loader = get_data_loaders(CONFIG['BATCH_SIZE'], CONFIG['nthreads'])
print(f"Training with {len(train_loader)} batches per epoch")

    # Initialize models
netG = Generator(CONFIG['noise_dim'], dp_rate=CONFIG['dp_rate']).to(DEVICE)
netD = Discriminator(std=CONFIG['gauss_std'], std_decay_rate=CONFIG['std_decay_rate']).to(DEVICE)

    # Apply weight initialization
netG.apply(weights_init)
netD.apply(weights_init)

In [ ]:
# Print model summaries
def print_model_summary(model, input_size):
  print(f"{model.__class__.__name__} Summary:")
  print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

        # Test forward pass
  if isinstance(input_size, tuple):
    x = torch.randn(1, *input_size).to(DEVICE)
  else:
    x = torch.randn(1, input_size).to(DEVICE)

  model.eval()
  with torch.no_grad():
    output = model(x)

  print(f"Input shape: {tuple(x.shape)}")
  print(f"Output shape: {tuple(output.shape)}")
  print("-" * 50)

# Print model summaries
print_model_summary(netG, CONFIG['noise_dim'])
print_model_summary(netD, (3, CONFIG['input_size'], CONFIG['input_size']))

In [ ]:
optG, optD, criterion, scaler = setup_training(netG, netD, CONFIG)

# Start training
print("Starting training...")
G_losses, D_losses = train(netG, netD, train_loader, criterion, optG, optD, scaler, CONFIG)

# Save final model
torch.save({
  'generator_state_dict': netG.state_dict(),
  'discriminator_state_dict': netD.state_dict(),
}, f"{CONFIG['checkpoint_dir']}/final_model.pt")

print("Training complete!")

In [ ]:
# Generate a final batch of samples
save_samples(CONFIG['num_epochs'], netG, DEVICE, 25, CONFIG['noise_dim'])

# Function to load and generate samples from a trained model
def generate_samples_from_checkpoint(checkpoint_path, n_samples=16):
        # Load the checkpoint
  checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

        # Recreate the generator and load the state dict
  netG = Generator(CONFIG['noise_dim'], dp_rate=CONFIG['dp_rate']).to(DEVICE)
  netG.load_state_dict(checkpoint['generator_state_dict'])
  netG.eval()

        # Generate samples
  with torch.no_grad():
    noise = torch.randn(n_samples, CONFIG['noise_dim'], device=DEVICE)
    fake_images = netG(noise)

            # Convert to numpy and denormalize
  fake_images = fake_images.detach().cpu().numpy()
  fake_images = np.transpose(fake_images, (0, 2, 3, 1))
  fake_images = (fake_images + 1) / 2.0
  fake_images = np.clip(fake_images, 0, 1)

    # Plot
  plt.figure(figsize=(16, 16))
  for i in range(n_samples):
    plt.subplot(int(np.sqrt(n_samples)), int(np.sqrt(n_samples)), i+1)
    plt.imshow(fake_images[i])
    plt.axis('off')

  plt.tight_layout()
  plt.savefig(f"{CONFIG['sample_dir']}/generated_samples.png")
  plt.show()

In [ ]:
!pip install pytorch-fid

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.linalg
import torchvision.models as models
from torch.utils.data import DataLoader

In [ ]:
class InceptionV3FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pre-trained InceptionV3 model
        self.inception = models.inception_v3(pretrained=True, transform_input=False, aux_logits=False)
        # Remove the final fully-connected layer
        self.inception.fc = nn.Identity()
        self.inception.eval()  # Set to evaluation mode

    def forward(self, x):
        # InceptionV3 expects 299x299 images; if not, resize outside this module.
        return self.inception(x)

In [ ]:
def calculate_fid(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Compute the FID score given two distributions."""
    diff = mu1 - mu2
    covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
         covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)
    return fid

def compute_statistics(features):
    """Compute the mean and covariance of a set of features."""
    mu = np.mean(features, axis=0)
    sigma = np.cov(features, rowvar=False)
    return mu, sigma

In [ ]:
def compute_fid_score(generator, real_loader, device, noise_dim, num_samples=1000):

    # Prepare the inception feature extractor
    inception = InceptionV3FeatureExtractor().to(device)
    inception.eval()

    # Containers for features
    real_features = []
    fake_features = []

    # -------------------------------
    # Extract features for real images
    # -------------------------------
    real_count = 0
    with torch.no_grad():
        for images, _ in real_loader:
            images = images.to(device)
            # Resize images to 299x299 for Inception
            images_resized = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
            feats = inception(images_resized)
            real_features.append(feats.cpu().numpy())
            real_count += images.size(0)
            if real_count >= num_samples:
                break
    real_features = np.concatenate(real_features, axis=0)
    real_features = real_features[:num_samples]

    # -------------------------------
    # Extract features for generated images
    # -------------------------------
    fake_count = 0
    with torch.no_grad():
        while fake_count < num_samples:
            current_batch = min(num_samples - fake_count, CONFIG['BATCH_SIZE'])
            noise = torch.randn(current_batch, noise_dim, device=device)
            fake_images = generator(noise)
            # Generator outputs images in [-1, 1]; convert to [0, 1]
            fake_images = (fake_images + 1) / 2.0
            # Resize to 299x299
            fake_images_resized = F.interpolate(fake_images, size=(299, 299), mode='bilinear', align_corners=False)
            feats = inception(fake_images_resized)
            fake_features.append(feats.cpu().numpy())
            fake_count += current_batch
    fake_features = np.concatenate(fake_features, axis=0)
    fake_features = fake_features[:num_samples]

    # -------------------------------
    # Compute statistics and FID score
    # -------------------------------
    mu_real, sigma_real = compute_statistics(real_features)
    mu_fake, sigma_fake = compute_statistics(fake_features)
    fid_value = calculate_fid(mu_real, sigma_real, mu_fake, sigma_fake)

    return fid_value

In [ ]:
fid_score = compute_fid_score(netG, real_loader, DEVICE, CONFIG['noise_dim'], num_samples=1000)
print(f"FID score: {fid_score:.2f}")